# 06 · Motor de microsimulación (Montecarlo, 2026-2075)

Proyecta la población base 2025 año a año hasta 2075, con 1.000 réplicas por escenario
(Base y Alternativo). Los agentes se agrupan en celdas con el mismo perfil y cada evento se
sortea con distribuciones binomiales sobre el tamaño de la celda. Cada año:

1. **Envejecimiento:** todos suman un año; quienes superan los 100 salen de la simulación.
2. **Nacimientos:** cascada empleo → pareja → empleo de la pareja → fecundidad → clima
   educativo del hogar del recién nacido, con los vectores de los notebooks 03-04. La paridad
   de las madres sube en uno.
3. **Educación:** el destino educativo máximo se sortea una sola vez por agente (vector de
   stock, según su cohorte y restringido a niveles ≥ al ya alcanzado); cada año se sortea la
   asistencia (vector de flujo condicionado al destino). El nivel avanza como máximo uno por año
   cursado, según la edad normativa, y nunca supera el destino.
4. **Recién nacidos:** sexo (`P_VARON = 0,509`) y mortalidad infantil por sexo.
5. **Mortalidad** de la población existente con el vector del notebook 01.

Las réplicas usan semillas fijas (`semilla = k`), de modo que cada corrida es reproducible y la
misma réplica `k` comparte semilla en los dos escenarios.

**Salidas por réplica** (`resultados/intermedios/simulaciones/{base,rebote}/`):
- `pop_kXXXX.parquet`: población total y por sexo × edad simple;
- `fert_kXXXX.parquet`: TGF sintética y nacimientos, paridad por cohorte de nacimiento y mujeres
  por edad × nivel educativo;
- `educ_kXXXX.parquet`: población por edad × nivel educativo, edad × asistencia y
  edad × asistencia × clima educativo de origen.

**Tiempo aproximado:** entre 30 y 45 segundos por réplica según la computadora, es decir
**8-12 horas por escenario y 17-25 horas en total** con 1.000 + 1.000 réplicas. Para una prueba rápida se puede fijar la
variable de entorno `K_SIMULACIONES` (por ejemplo, `K_SIMULACIONES=5`). Si se interrumpe, al
volver a correrlo retoma desde la primera réplica que no está en disco.

In [ ]:
import gc
import os
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
DIR_VEC = Path("resultados/intermedios/vectores")
DIR_SIM = Path(os.environ.get("DIR_SIMULACIONES", "resultados/intermedios/simulaciones"))
RUTA_POB = Path("resultados/intermedios/poblacion_base/poblacion_base.csv")

K_SIMULACIONES = int(os.environ.get("K_SIMULACIONES", 1000))
ESCENARIOS = {"base": "fertility_vector.parquet", "rebote": "fertility_vector_rebote.parquet"}
N_ANIOS = 50
YEAR_BASE = 2025
SEMILLA_BASE = 0

P_VARON = 0.509
PROVINCIA_REGION = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3, 12: 3,
                    13: 4, 14: 4, 15: 4, 16: 5, 17: 5, 18: 5, 19: 5, 20: 6, 21: 6, 22: 6, 23: 6, 24: 6}
STOCK_COLS = [f"prob_max_level{i}" for i in range(7)]
UMBRALES_ANIOS = np.array([0, 1, 5, 6, 10, 11, 16], dtype=np.int16)
ANIOS_MAX_CURSADOS = int(UMBRALES_ANIOS[-1])
EDADES_NIVEL = np.array([0, 6, 12, 13, 18, 19, 23], dtype=np.int16)
EDAD_FIN_ETAPA = np.array([-1, 11, -1, 17, -1, 22, -1], dtype=np.int16)
EDAD_MIN_EDUCACION, EDAD_MAX_EDUCACION = 3, 80
PARITY_MAX = 10
PARITY_MAX_VECTOR = 7
DTYPES_ESTADO = {"sex": np.int8, "status": np.int8, "provincia": np.int8, "education": np.int8,
                 "age": np.int8, "clima_educativo": np.int8, "generation": np.int8, "region": np.int8,
                 "parity": np.int8, "in_system": np.int8, "max_level_target": np.int8,
                 "years_in_system": np.int8, "birth_year": np.int16, "n": np.int64}

mortality_vector = pd.read_parquet(DIR_VEC / 'mortality_vector.parquet')
employment_vector = pd.read_parquet(DIR_VEC / 'employment_vector.parquet')
partner_vector = pd.read_parquet(DIR_VEC / 'partner_vector.parquet')
education_stock_vector = pd.read_parquet(DIR_VEC / 'education_stock_vector.parquet')
education_flow_vector = pd.read_parquet(DIR_VEC / 'education_flow_vector.parquet')

## Funciones auxiliares

In [ ]:
def get_generation(birth_year):
    birth_year = pd.Series(birth_year) if np.isscalar(birth_year) else birth_year
    conditions = [birth_year.between(1946, 1964), birth_year.between(1965, 1980),
                  birth_year.between(1981, 1996), birth_year.between(1997, 2012),
                  birth_year.between(2013, 2025), birth_year.between(2026, 2039),
                  birth_year.between(2040, 2054), birth_year.between(2055, 2069),
                  birth_year.between(2070, 2084)]
    result = np.select(conditions, [1, 2, 3, 4, 5, 6, 7, 8, 9], default=0)
    return result[0] if len(result) == 1 else result


def nivel_por_edad(edad):
    return (np.searchsorted(EDADES_NIVEL, np.asarray(edad, dtype=np.int16), side="right") - 1).astype(np.int8)


def sigue_en_etapa(education, max_level_target, age):
    """True si el agente alcanzó un destino «en curso» (1, 3 o 5) pero sigue en la edad de esa etapa."""
    edu = np.asarray(education, dtype=np.int16)
    tgt = np.clip(np.asarray(max_level_target, dtype=np.int16), 0, 6)
    fin = EDAD_FIN_ETAPA[tgt]
    return (edu >= tgt) & (fin >= 0) & (np.asarray(age, dtype=np.int16) <= fin)


def sample_categorical(probs, rng, fallback=0):
    probs = np.asarray(probs, dtype=float)
    row_sums = probs.sum(axis=1, keepdims=True)
    filas_invalidas = row_sums.ravel() <= 0
    cumprobs = np.cumsum(probs / np.where(row_sums <= 0, 1.0, row_sums), axis=1)
    cumprobs[:, -1] = 1.0
    categoria = np.clip((rng.random((probs.shape[0], 1)) > cumprobs).sum(axis=1), 0, probs.shape[1] - 1)
    return np.where(filas_invalidas, fallback, categoria)


def merge_seguro(izq, der, **kwargs):
    out = izq.merge(der, **kwargs)
    assert len(out) == len(izq), "un merge cambió la cantidad de filas (clave duplicada)"
    return out

## Eventos del año

In [ ]:
def assign_natality(poblacion_base, current_year, rng, fertility_vector):
    women = poblacion_base.loc[(poblacion_base["sex"] == 1) & (poblacion_base["age"] >= 15)
                               & (poblacion_base["age"] < 50) & (poblacion_base["n"] > 0)].copy()
    cols_merge = ["sex", "provincia", "status", "age", "education"]
    women = merge_seguro(women, employment_vector[cols_merge + [f"prob_work_{current_year}"]],
                         on=cols_merge, how="left").rename(columns={f"prob_work_{current_year}": "p_employ"})
    women = merge_seguro(women, partner_vector[cols_merge + [f"prob_couple_{current_year}", f"prob_swork_{current_year}"]],
                         on=cols_merge, how="left").rename(columns={f"prob_couple_{current_year}": "p_partner",
                                                                    f"prob_swork_{current_year}": "p_partner_employed"})
    women[["p_employ", "p_partner", "p_partner_employed"]] = women[["p_employ", "p_partner", "p_partner_employed"]].fillna(0)

    n_tot = women["n"].round().astype(int).values
    n_employed = rng.binomial(n_tot, women["p_employ"].values)
    n_unemployed = n_tot - n_employed
    n_emp_couple = rng.binomial(n_employed, women["p_partner"].values)
    n_emp_nocouple = n_employed - n_emp_couple
    n_unemp_couple = rng.binomial(n_unemployed, women["p_partner"].values)
    n_unemp_nocouple = n_unemployed - n_unemp_couple
    n_emp_couple_pwork = rng.binomial(n_emp_couple, women["p_partner_employed"].values)
    n_emp_couple_nopwork = n_emp_couple - n_emp_couple_pwork
    n_unemp_couple_pwork = rng.binomial(n_unemp_couple, women["p_partner_employed"].values)
    n_unemp_couple_nopwork = n_unemp_couple - n_unemp_couple_pwork
    women = pd.concat([women.assign(hh_work=0, couple=0, n=n_unemp_nocouple),
                       women.assign(hh_work=0, couple=1, n=n_unemp_couple_nopwork),
                       women.assign(hh_work=1, couple=1, n=n_unemp_couple_pwork),
                       women.assign(hh_work=1, couple=0, n=n_emp_nocouple),
                       women.assign(hh_work=1, couple=1, n=(n_emp_couple_nopwork + n_emp_couple_pwork))],
                      ignore_index=True)
    women = women[women["n"] > 0].drop(columns=["p_employ", "p_partner", "p_partner_employed"])

    cols_fert = ["sex", "provincia", "status", "age", "education", "couple", "hh_work", "parity"]
    women["_parity_real"] = women["parity"].values
    women["parity"] = np.minimum(women["parity"].values, PARITY_MAX_VECTOR)
    women = merge_seguro(women, fertility_vector[cols_fert + [f"prob_{current_year}"]], on=cols_fert, how="left")
    women["parity"] = women["_parity_real"].values
    women = women.drop(columns=["_parity_real"])
    women[f"prob_{current_year}"] = women[f"prob_{current_year}"].fillna(0)
    women[f"births_{current_year}"] = rng.binomial(women["n"].values, women[f"prob_{current_year}"].values)

    cedu_cols = [f"prob_cedu{i}_{current_year}" for i in range(5)]
    women_cedu = merge_seguro(women, partner_vector[cols_merge + cedu_cols], on=cols_merge, how="left")
    women_cedu[cedu_cols] = women_cedu[cedu_cols].fillna(0)
    remaining, probs, dfs_cedu = women_cedu[f"births_{current_year}"].values.copy(), women_cedu[cedu_cols].values, []
    for i in range(5):
        p_i = np.clip(probs[:, i] / np.clip(probs[:, i:].sum(axis=1), 1e-9, None), 0, 1)
        n_i = rng.binomial(remaining, p_i)
        remaining = remaining - n_i
        dfs_cedu.append(women_cedu.assign(clima_educativo=i, **{f"births_{current_year}": n_i}))
    births_cedu = pd.concat(dfs_cedu, ignore_index=True)
    births_cedu = births_cedu[births_cedu[f"births_{current_year}"] > 0]

    births = births_cedu.groupby(["provincia", "education", "status", "age", "clima_educativo"],
                                 as_index=False)[f"births_{current_year}"].sum()
    births["birth_year"] = (current_year - births["age"].astype(np.int16)).astype(np.int16)
    births["generation"] = get_generation(births["birth_year"])
    return births, women


def assign_mortality(df, current_year, rng):
    df = df[df["n"] > 0].copy()
    col_qx = f"qx_{current_year}"
    df = merge_seguro(df, mortality_vector[["provincia", "age", "sex", col_qx]], on=["provincia", "age", "sex"], how="left")
    df["n"] = rng.binomial(df["n"].fillna(0).round().astype(int).values,
                           np.clip((1 - df[col_qx].fillna(0)).values, 0, 1))
    return df.drop(columns=[col_qx])


def assign_infant_mortality(df, current_year, rng, col_nac):
    df_con_nac, df_sin_nac = df[df[col_nac] > 0].copy(), df[df[col_nac] <= 0].copy()
    col_qx = f"qx_{current_year}"
    df_con_nac = merge_seguro(df_con_nac, mortality_vector.loc[mortality_vector["age"] == 0, ["provincia", "sex", col_qx]],
                              on=["provincia", "sex"], how="left")
    df_con_nac[col_nac] = rng.binomial(df_con_nac[col_nac].round().astype(int).values,
                                       np.clip((1 - df_con_nac[col_qx].fillna(0)).values, 0, 1))
    return pd.concat([df_sin_nac, df_con_nac.drop(columns=[col_qx])], ignore_index=True)


def assign_education(df, current_year, rng):
    mask_edad = (df["age"] >= EDAD_MIN_EDUCACION) & (df["age"] <= EDAD_MAX_EDUCACION) & (df["n"] > 0)
    df_rest, df_edu = df.loc[~mask_edad].copy(), df.loc[mask_edad].copy()
    df_rest["in_system"] = 0
    if df_edu.empty:
        return df_rest

    # (a) destino educativo: se sortea una vez, según la cohorte, y nunca por debajo del nivel actual
    mask_sin_destino = df_edu["max_level_target"] < 0
    if mask_sin_destino.any():
        df_sin_destino, df_con_destino = df_edu.loc[mask_sin_destino].copy(), df_edu.loc[~mask_sin_destino].copy()
        cols_stock = ["sex", "provincia", "status", "clima_educativo"]
        df_sin_destino = merge_seguro(df_sin_destino, education_stock_vector[cols_stock + ["cohorte"] + STOCK_COLS],
                                      left_on=cols_stock + ["birth_year"], right_on=cols_stock + ["cohorte"], how="left")
        df_sin_destino = df_sin_destino.drop(columns=["cohorte"])
        df_sin_destino[STOCK_COLS] = df_sin_destino[STOCK_COLS].fillna(0)
        P = df_sin_destino[STOCK_COLS].to_numpy(dtype=float)
        edu = df_sin_destino["education"].to_numpy(dtype=int)
        P = np.where(np.arange(P.shape[1])[None, :] >= edu[:, None], P, 0.0)
        techo = np.where(P.sum(axis=1) <= 0)[0]
        P[techo, np.clip(edu[techo], 0, P.shape[1] - 1)] = 1.0
        df_sin_destino["max_level_target"] = sample_categorical(P, rng, fallback=0).astype(np.int8)
        df_edu = pd.concat([df_con_destino, df_sin_destino.drop(columns=STOCK_COLS)], ignore_index=True)

    # (b) asistencia del año, sólo para quien todavía no alcanzó su destino (o lo cursa)
    mask_con_margen = ((df_edu["education"] < df_edu["max_level_target"])
                       | sigue_en_etapa(df_edu["education"].values, df_edu["max_level_target"].values,
                                        df_edu["age"].values))
    df_sin_margen = df_edu.loc[~mask_con_margen].copy()
    df_sin_margen["in_system"] = np.int8(0)
    df_edu = df_edu.loc[mask_con_margen].copy()
    if df_edu.empty:
        df_final = pd.concat([df_rest, df_sin_margen], ignore_index=True)
        return df_final[df_final["n"] > 0]

    cols_flow = ["sex", "provincia", "status", "age", "clima_educativo", "max_level_target"]
    col_prob = f"prob_inschool_{current_year}"
    df_edu = merge_seguro(df_edu, education_flow_vector[cols_flow + [col_prob]], on=cols_flow, how="left")
    df_edu[col_prob] = df_edu[col_prob].fillna(0)
    n_tot = df_edu["n"].round().astype(np.int64).values
    n_stay = rng.binomial(n_tot, df_edu[col_prob].values)
    df_edu = df_edu.drop(columns=[col_prob])
    df_drop = df_edu.assign(n=n_tot - n_stay, in_system=np.int8(0))
    df_drop = df_drop[df_drop["n"] > 0]
    df_stay = df_edu.assign(n=n_stay, in_system=np.int8(1))
    df_stay = df_stay[df_stay["n"] > 0]

    # (c) quien cursó suma un año y avanza a lo sumo un nivel (edad normativa, tope en el destino)
    if not df_stay.empty:
        df_stay["years_in_system"] = np.minimum(df_stay["years_in_system"].values.astype(np.int16) + 1,
                                                ANIOS_MAX_CURSADOS).astype(np.int8)
        edu = df_stay["education"].values.astype(np.int16)
        tgt = df_stay["max_level_target"].values.astype(np.int16)
        avanza = ((edu < nivel_por_edad(df_stay["age"].values)) & (edu < tgt)).astype(np.int16)
        df_stay["education"] = np.minimum(edu + avanza, tgt).astype(np.int8)
        egresa = ((df_stay["education"].values >= df_stay["max_level_target"].values)
                  & ~sigue_en_etapa(df_stay["education"].values, df_stay["max_level_target"].values,
                                    df_stay["age"].values))
        df_stay.loc[egresa, "in_system"] = np.int8(0)

    df_edu_final = pd.concat([df_sin_margen, df_drop, df_stay], ignore_index=True)
    df_edu_final = df_edu_final.groupby([c for c in df_edu_final.columns if c != "n"], as_index=False,
                                        sort=False, dropna=False)["n"].sum()
    df_final = pd.concat([df_rest, df_edu_final], ignore_index=True)
    return df_final[df_final["n"] > 0]

## Estadísticas que se exportan

In [ ]:
def filas_grupo(serie, k, year, grupo, col="n"):
    out = serie.reset_index()
    claves = [c for c in out.columns if c != serie.name]
    out["categoria"] = out[claves].astype(str).agg("_".join, axis=1)
    out = out.rename(columns={serie.name: col})
    out["k"], out["year"], out["grupo"] = k, year, grupo
    return out[["k", "year", "grupo", "categoria", col]].to_dict("records")


def get_popstats(df, year, k):
    filas = [{"k": k, "year": year, "grupo": "total", "categoria": "total", "n": df["n"].sum()}]
    filas += filas_grupo(df.groupby(["sex", "age"])["n"].sum(), k, year, "sex_x_age")
    return filas


def get_fertilitystats(women_df, births_df, year, k, births_col):
    filas = []
    births = births_df[births_col].sum()
    w_age = women_df.groupby("age")["n"].sum()
    b_age = births_df.groupby("age")[births_col].sum()
    asfr = (b_age.reindex(w_age.index).fillna(0) / w_age.replace(0, np.nan)).dropna()
    asfr = asfr[(asfr.index >= 15) & (asfr.index <= 49)]
    filas.append({"k": k, "year": year, "grupo": "tgf_sintetica", "categoria": "total",
                  "women_n": float(women_df["n"].sum()), "births_n": births, "children_n": 0.0,
                  "tgf_sintetica": float(asfr.sum())})

    women_df = women_df.assign(parity_x_n=women_df["parity"] * women_df["n"])
    comb = pd.concat([women_df.groupby(["age", "education"])["n"].sum(),
                      women_df.groupby(["age", "education"])["parity_x_n"].sum(),
                      births_df.groupby(["age", "education"])[births_col].sum()],
                     axis=1, keys=["women_n", "parity_n", "births_n"]).fillna(0).reset_index()
    comb["categoria"] = comb[["age", "education"]].astype(str).agg("_".join, axis=1)
    comb["children_n"] = comb["parity_n"] + comb["births_n"]
    comb = comb.assign(k=k, year=year, grupo="age_x_education")
    filas += comb[["k", "year", "grupo", "categoria", "women_n", "births_n", "children_n"]].to_dict("records")

    coh = women_df.groupby(["birth_year", "parity"], dropna=False, as_index=False)["n"].sum()
    coh = coh[coh["n"] > 0].copy()
    coh["categoria"] = coh["birth_year"].astype("Int64").astype(str) + "_" + coh["parity"].astype(str)
    coh = coh.assign(k=k, year=year, grupo="cohorte_birth_year_x_parity", women_n=coh["n"], births_n=0.0,
                     children_n=coh["parity"] * coh["n"])
    filas += coh[["k", "year", "grupo", "categoria", "women_n", "births_n", "children_n"]].to_dict("records")
    return filas


def get_educstats(df, year, k):
    df_esc = df[(df["age"] >= EDAD_MIN_EDUCACION) & (df["age"] <= EDAD_MAX_EDUCACION)]
    filas = filas_grupo(df.groupby(["age", "education"])["n"].sum(), k, year, "age_x_education")
    filas += filas_grupo(df_esc.groupby(["age", "in_system"])["n"].sum(), k, year, "age_x_in_system")
    filas += filas_grupo(df_esc.groupby(["age", "in_system", "clima_educativo"])["n"].sum(), k, year,
                         "age_x_in_system_x_clima_educativo")
    return filas


def exportar(filas, ruta):
    df = pd.DataFrame(filas)
    conteo = [c for c in df.columns if c not in ("grupo", "categoria", "tgf_sintetica")]
    df[conteo] = df[conteo].fillna(0)
    df[["grupo", "categoria"]] = df[["grupo", "categoria"]].astype(str)
    df.to_parquet(ruta, index=False)

## Una réplica Montecarlo

In [ ]:
def simular_replica(k, fertility_vector, dir_salida):
    rng = np.random.default_rng(seed=SEMILLA_BASE + k)
    poblacion = pd.read_csv(RUTA_POB, sep=",")
    poblacion["parity"] = poblacion["children"].clip(upper=PARITY_MAX).astype(np.int8)
    poblacion = poblacion.groupby([c for c in poblacion.columns if c not in ["n", "children"]],
                                  as_index=False)["n"].sum()
    poblacion["in_system"] = np.int8(0)
    poblacion["max_level_target"] = np.int8(-1)
    poblacion["years_in_system"] = UMBRALES_ANIOS[poblacion["education"].clip(0, 6).astype(int).values].astype(np.int8)
    for c, t in DTYPES_ESTADO.items():
        if c in poblacion.columns:
            poblacion[c] = poblacion[c].astype(t)

    claves_fr = ["provincia", "education", "status", "age"]
    fertility_rate = poblacion.loc[(poblacion["sex"] == 1) & (poblacion["age"] >= 15) & (poblacion["age"] < 50),
                                   claves_fr].drop_duplicates().copy()

    pop_rows, fert_rows, educ_rows = [], [], []
    for anio_i in range(1, N_ANIOS + 1):
        current_year = YEAR_BASE + anio_i
        col_nac = f"births_{current_year}"
        n_inicio = int(poblacion["n"].sum())
        n_truncado = int(poblacion.loc[poblacion["age"] > 100, "n"].sum())
        poblacion = poblacion[(poblacion["age"] <= 100) & (poblacion["n"] > 0)].copy()
        poblacion["year"] = current_year
        poblacion["age"] = (poblacion["age"] + 1).astype(np.int8)

        # nacimientos y paridad de las madres
        births, women_detail = assign_natality(poblacion, current_year, rng, fertility_vector)
        id_cols = [c for c in poblacion.columns if c != "n" and c in women_detail.columns]
        movidas = women_detail.groupby(id_cols, as_index=False)[col_nac].sum()
        movidas = movidas[movidas[col_nac] > 0].rename(columns={col_nac: "n_movido"})
        poblacion = merge_seguro(poblacion, movidas, on=id_cols, how="left")
        poblacion["n_movido"] = poblacion["n_movido"].fillna(0).astype(int)
        poblacion["n"] = (poblacion["n"] - poblacion["n_movido"]).clip(lower=0)
        destino = movidas.copy()
        destino["parity"] = (destino["parity"] + 1).clip(upper=PARITY_MAX).astype(np.int8)
        poblacion = pd.concat([poblacion.drop(columns="n_movido"), destino.rename(columns={"n_movido": "n"})],
                              ignore_index=True)
        poblacion = poblacion.groupby([c for c in poblacion.columns if c != "n"], as_index=False, dropna=False)["n"].sum()
        poblacion = poblacion[poblacion["n"] > 0]

        # matriz de nacimientos por celda materna (acumula todos los años)
        fertility_rate = fertility_rate.merge(births.groupby(claves_fr, as_index=False)[col_nac].sum(),
                                              on=claves_fr, how="outer")
        cols_nac = [c for c in fertility_rate.columns if c.startswith("births_")]
        fertility_rate[cols_nac] = fertility_rate[cols_nac].fillna(0)

        poblacion = assign_education(poblacion, current_year, rng)

        # recién nacidos: sexo y mortalidad infantil
        newborns = births.groupby(["provincia", "education", "status", "clima_educativo"], as_index=False)[col_nac].sum()
        nac_tot = newborns[col_nac].round().astype(int)
        newborns["n_varones"] = rng.binomial(nac_tot.values, P_VARON)
        newborns["n_mujeres"] = nac_tot - newborns["n_varones"]
        newborns = pd.concat(
            [newborns.drop(columns=[col_nac, "n_varones"]).rename(columns={"n_mujeres": col_nac}).assign(sex=1),
             newborns.drop(columns=[col_nac, "n_mujeres"]).rename(columns={"n_varones": col_nac}).assign(sex=0)],
            ignore_index=True)
        newborns = assign_infant_mortality(newborns, current_year, rng, col_nac).rename(columns={col_nac: "n"})
        newborns = newborns[newborns["n"] > 0].copy()
        newborns["education"] = np.int8(0)
        newborns["years_in_system"] = np.int8(0)
        newborns["age"] = np.int8(0)
        newborns["parity"] = np.int8(0)
        newborns["region"] = newborns["provincia"].map(PROVINCIA_REGION).astype(np.int8)
        newborns["generation"] = np.int8(get_generation(current_year))
        newborns["year"] = current_year
        newborns["in_system"] = np.int8(0)
        newborns["max_level_target"] = np.int8(-1)
        newborns["birth_year"] = np.int16(current_year)

        # mortalidad de la población existente y cierre del año
        n_antes = int(poblacion["n"].sum())
        poblacion = assign_mortality(poblacion, current_year, rng)
        defunciones = n_antes - int(poblacion["n"].sum())
        poblacion = pd.concat([poblacion, newborns], ignore_index=True)
        poblacion = poblacion.groupby([c for c in poblacion.columns if c != "n"], as_index=False, dropna=False)["n"].sum()
        assert int(poblacion["n"].sum()) == n_inicio - n_truncado - defunciones + int(newborns["n"].sum()), \
            f"la identidad poblacional no cierra en {current_year}"
        poblacion = poblacion[poblacion["n"] > 0].reset_index(drop=True)

        pop_rows += get_popstats(poblacion, current_year, k)
        fert_rows += get_fertilitystats(women_detail, fertility_rate, current_year, k, col_nac)
        educ_rows += get_educstats(poblacion, current_year, k)

    for nombre, filas in (("pop", pop_rows), ("fert", fert_rows), ("educ", educ_rows)):
        exportar(filas, dir_salida / f"{nombre}_k{k:04d}.parquet")
    gc.collect()

## Corrida de los dos escenarios

In [ ]:
for escenario, archivo_vector in ESCENARIOS.items():
    dir_salida = DIR_SIM / escenario
    dir_salida.mkdir(parents=True, exist_ok=True)
    fertility_vector = pd.read_parquet(DIR_VEC / archivo_vector)
    for k in range(1, K_SIMULACIONES + 1):
        if (dir_salida / f"educ_k{k:04d}.parquet").exists():
            continue
        simular_replica(k, fertility_vector, dir_salida)
        if k % 10 == 0 or k == K_SIMULACIONES:
            print(f"[{escenario}] réplica {k}/{K_SIMULACIONES} terminada", flush=True)
    del fertility_vector
    gc.collect()
print("simulaciones terminadas")